# GNN grid comparison (run 11, phase 2)

The diagrams of the per-run GNN notebooks (`data/run11-gnn-*/figures.ipynb`), with the selected
GNN variants in one diagram each. Choose the variants in the next cell: `True` to include,
`False` to leave out.

Encoding:
- colour = architecture (convolution × depth);
- line style = objective (solid: cosine; dashed: cosine + 0.05·InfoNCE);
- grey dotted = plain reference mean, the same for every variant;
- ring = each variant's selected epoch (best validation cosine).

Figures go to `figures/gnn_comparison/` as PDF and PNG, drawn at their printed size.
Re-running with a different selection overwrites them, so copy the files you keep into the
thesis folder.

In [ ]:
INCLUDE = {
    "gatv2-l1-cos":    True,
    "gatv2-l1-cosnce": True,
    "gatv2-l2-cos":    True,
    "gatv2-l2-cosnce": True,
    "sage-l1-cos":     True,
    "sage-l1-cosnce":  True,
    "sage-l2-cos":     True,
    "sage-l2-cosnce":  True,
}

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, PercentFormatter

DIAGRAMS = Path.cwd()                         # diagrams/
sys.path.insert(0, str(DIAGRAMS))
import thesis_style as ts

ts.apply()
# Printed size of every figure. Taller than the single-run figures: the legend lists up to
# eight variants. ts.TEXTWIDTH = full width, ts.HALFWIDTH = two subfigures side by side.
ts.FIGWIDTH, ts.FIGHEIGHT = 0.75 * ts.TEXTWIDTH, 85 * ts.MM
C = ts.COLORS
OUT = DIAGRAMS / "figures" / "gnn_comparison"
JOB = "14541449"

ARCH_COLOR = {("gatv2", 1): C["blue"], ("gatv2", 2): C["sky"],
              ("sage", 1): C["vermillion"], ("sage", 2): C["orange"]}
CONV = {"gatv2": "GATv2", "sage": "GraphSAGE"}

runs = {}
baselines = []
for short, keep in INCLUDE.items():
    if not keep:
        continue
    run_dir = DIAGRAMS / "data" / f"run11-gnn-{short}-{JOB}"
    config = json.loads((run_dir / "config.json").read_text())
    history = pd.read_csv(run_dir / "history.csv")
    base = history.filter(like="pretrain/refs_mean_").dropna(how="all").iloc[0]
    base.index = base.index.str.removeprefix("pretrain/refs_mean_")
    baselines.append(base)
    log = history.dropna(subset=["pretrain/epoch"]).copy()
    log.columns = log.columns.str.removeprefix("pretrain/")
    log["epoch"] = log["epoch"].astype(int) + 1
    eligible = log[log["output_cosine"] <= config["cosine_gate"]]
    log["objective"] = (config["lambda_cos"] * (1 - log.train_cosine)
                        + config["lambda_nce"] * log.loss
                        + config["vicreg_var"] * log.variance
                        + config["vicreg_cov"] * log.covariance)
    objective_name = "cos" if config["lambda_nce"] == 0 else "cos+InfoNCE"
    runs[short] = dict(
        log=log, best=eligible.loc[eligible["mean_cosine"].idxmax()],
        color=ARCH_COLOR[(config["gnn_conv"], config["gnn_layers"])],
        ls="-" if config["lambda_nce"] == 0 else "--",
        label=f"{CONV[config['gnn_conv']]}, {config['gnn_layers']} layer"
              f"{'s' if config['gnn_layers'] > 1 else ''}, {objective_name}")
if not runs:
    raise SystemExit("INCLUDE selects no run")

# The baseline is measured on the same validation papers by every run, so it must agree.
baseline = baselines[0]
assert all(np.allclose(b.values, baseline.values) for b in baselines), "baselines differ"

for short, r in runs.items():
    b = r["best"]
    print(f"{r['label']:<36} selected epoch {int(b.epoch):>2}: cosine {b.mean_cosine:.4f} "
          f"({b.mean_cosine - baseline.mean_cosine:+.4f} vs reference mean), top-1 {b.top1:.1%}")
print(f"{'reference mean':<36} cosine {baseline.mean_cosine:.4f}, top-1 {baseline.top1:.1%}")

def plot_metric(key, ylabel, name, pct=False, with_baseline=True, with_selected=True, ylim=None):
    # One W&B metric over epochs for every included variant.
    scale = 100 if pct else 1
    fig, ax = ts.figure()
    for r in runs.values():
        ax.plot(r["log"].epoch, scale * r["log"][key], color=r["color"], ls=r["ls"],
                label=r["label"])
        if with_selected:
            ax.plot(r["best"].epoch, scale * r["best"][key], ls="none", marker="o", ms=6,
                    mfc="none", mec=r["color"], mew=0.9, zorder=3)
    if with_baseline and key in baseline.index:
        ax.axhline(scale * baseline[key], color=C["grey"], ls=":", lw=1.4, label="Reference mean")
    epochs = max(r["log"].epoch.max() for r in runs.values())
    ax.set(xlabel="Epoch", ylabel=ylabel, xlim=(0.5, epochs + 0.5))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    if pct:
        ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))
    if ylim is not None:
        ax.set_ylim(*ylim)
    ts.legend_below(fig, ncol=2)
    ts.save(fig, f"gnn_{name}", OUT)
    plt.show()

## Validation cosine to the paper's own embedding

*Suggested caption:* Cosine similarity between the GNN output and the paper's own embedding on
the 806 validation papers, per GNN variant. Rings: selected epoch. Dotted: plain mean of the
references' embeddings.

In [ ]:
plot_metric("mean_cosine", "Cosine to own embedding", "cosine_validation")

## Training cosine

*Suggested caption:* Cosine similarity between GNN output and own embedding on training
batches, per GNN variant.

In [ ]:
plot_metric("train_cosine", "Cosine to own embedding", "cosine_training",
            with_baseline=False, with_selected=False)

## Training objective

Each variant's own objective, so the cosine-only and InfoNCE variants are on different scales.

*Suggested caption:* Training objective per GNN variant.

In [ ]:
plot_metric("objective", "Training loss", "objective", with_baseline=False,
            with_selected=False)

## Retrieval among the 806 validation papers: top-1 and top-5

*Suggested caption:* Top-1 [top-5] retrieval of the paper's own embedding among 806 validation
papers, per GNN variant. Dotted: plain reference mean.

In [ ]:
plot_metric("top1", "Top-1 retrieval", "top1", pct=True)

In [ ]:
plot_metric("top5", "Top-5 retrieval", "top5", pct=True)

## Median rank

*Suggested caption:* Median retrieval rank of the paper's own embedding among 806 validation
papers, per GNN variant.

In [ ]:
plot_metric("median_rank", "Median rank", "median_rank",
            ylim=(0, max(2, max(r["log"].median_rank.max() for r in runs.values())) + 1))

## Output diversity

*Suggested caption:* Mean pairwise cosine between GNN outputs of different validation papers,
per GNN variant. Dotted: plain reference mean.

In [ ]:
plot_metric("output_cosine", "Output pairwise cosine", "output_cosine", ylim=(0, 1))

## InfoNCE loss and in-batch accuracy (training)

Monitored for every variant; optimised only by the InfoNCE variants (dashed).

*Suggested caption:* InfoNCE loss [in-batch accuracy] on training batches, per GNN variant.

In [ ]:
plot_metric("loss", "InfoNCE (training)", "infonce", with_baseline=False, with_selected=False)

In [ ]:
plot_metric("batch_accuracy", "In-batch accuracy (training)", "batch_accuracy", pct=True,
            with_baseline=False, with_selected=False)

## VICReg terms (training, diagnostics only; weight 0 in run 11)

*Suggested caption:* VICReg variance [covariance] term of the GNN outputs on training batches,
per GNN variant; not part of the run-11 objective.

In [ ]:
plot_metric("variance", "Variance term", "vicreg_variance", with_baseline=False,
            with_selected=False)

In [ ]:
plot_metric("covariance", "Covariance term", "vicreg_covariance", with_baseline=False,
            with_selected=False)